In [ ]:
# 1. Setup & Imports
!pip install -q transformers datasets scikit-learn accelerate pandas seaborn matplotlib

import torch
import pandas as pd
import numpy as np
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Check GPU
device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
print(f"🔥 Using Device: {device_name}")
if torch.cuda.is_available():
    print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ WARNING: GPU not detected! Enable GPU in Settings.")

In [ ]:
# 2. Load Data from Kaggle Input
# Upload your CSV file to Kaggle → Add Data → Upload
# Ganti path sesuai nama file yang di-upload

# Option 1: Jika upload manual
file_path = '/kaggle/input/your-dataset-name/gojek_FINAL_3class_BALANCED.csv'

# Option 2: Jika dari Kaggle Dataset (lebih recommended)
# file_path = '/kaggle/input/dataset-name/file.csv'

print("📂 Loading dataset...")
df = pd.read_csv(file_path)

print(f"✅ Dataset loaded: {len(df):,} rows")
print(f"📊 Columns: {list(df.columns)}")

# Show sample
print("\n📋 Sample data:")
print(df.head())

# Distribution
print("\n📊 Sentiment Distribution:")
print(df['sentiment'].value_counts())

In [ ]:
# 3. Data Preprocessing
# Label Mapping
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
df['label'] = df['sentiment'].map(label_map)

# Split Data (Stratified to keep balance)
# 80% Train, 10% Validation, 10% Test
print("\n🔀 Splitting data...")
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['text'].tolist(), 
    df['label'].tolist(), 
    test_size=0.2, 
    stratify=df['label'], 
    random_state=42
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, 
    temp_labels, 
    test_size=0.5, 
    stratify=temp_labels, 
    random_state=42
)

print(f"✅ Train Size: {len(train_texts):,}")
print(f"✅ Val Size:   {len(val_texts):,}")
print(f"✅ Test Size:  {len(test_texts):,}")

In [ ]:
# 4. Tokenization
print("\n🔤 Loading tokenizer...")
model_name = 'indobenchmark/indobert-base-p1'
tokenizer = BertTokenizer.from_pretrained(model_name)

def tokenize_data(texts, labels):
    encodings = tokenizer(texts, truncation=True, padding=True, max_length=128)
    dataset = []
    for i in range(len(texts)):
        item = {key: torch.tensor(val[i]) for key, val in encodings.items()}
        item['labels'] = torch.tensor(labels[i])
        dataset.append(item)
    return dataset

print("⏳ Tokenizing train data...")
train_dataset = tokenize_data(train_texts, train_labels)

print("⏳ Tokenizing validation data...")
val_dataset = tokenize_data(val_texts, val_labels)

print("⏳ Tokenizing test data...")
test_dataset = tokenize_data(test_texts, test_labels)

print("✅ Tokenization complete!")

In [ ]:
# 5. Metrics Function
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

print("✅ Metrics function ready")

In [ ]:
# 6. Model Configuration
print("\n🤖 Loading IndoBERT model...")

id2label = {0: 'negative', 1: 'neutral', 2: 'positive'}
label2id = {'negative': 0, 'neutral': 1, 'positive': 2}

model = BertForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=3, 
    id2label=id2label, 
    label2id=label2id
)

print("✅ Model loaded successfully!")
print(f"📊 Model size: {sum(p.numel() for p in model.parameters()):,} parameters")

In [ ]:
# 7. Training Arguments (Optimized for Kaggle T4 GPU)
training_args = TrainingArguments(
    output_dir='./results_3class',
    num_train_epochs=5,              # Max epochs
    per_device_train_batch_size=16,  # Batch size for T4 (16GB VRAM)
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,   # Virtual batch size = 32
    learning_rate=2e-5,              # Low LR to prevent overfitting
    weight_decay=0.01,               # Regularization
    warmup_ratio=0.1,
    eval_strategy="epoch",           # Evaluate every epoch
    save_strategy="epoch",
    load_best_model_at_end=True,     # Load best model
    metric_for_best_model="accuracy",
    fp16=True,                       # Mixed Precision for faster training
    logging_dir='./logs',
    logging_steps=50,
    report_to="none",                # Disable wandb on Kaggle
    dataloader_num_workers=2         # Kaggle supports multiprocessing
)

print("✅ Training configuration ready")

In [ ]:
# 8. Initialize Trainer
print("\n🎯 Initializing Trainer...")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]  # Stop if no improve for 2 epochs
)

print("✅ Trainer ready!")

In [ ]:
# 9. START TRAINING 🚀
print("\n" + "="*80)
print("🚀 STARTING TRAINING")
print("="*80)

trainer.train()

print("\n" + "="*80)
print("✅ TRAINING COMPLETE!")
print("="*80)

In [ ]:
# 10. Evaluation on Test Set
print("\n📊 Evaluating on Test Set...")
test_result = trainer.predict(test_dataset)

print("\n✅ Test Results:")
for key, value in test_result.metrics.items():
    print(f"   {key}: {value:.4f}")

In [ ]:
# 11. Detailed Classification Report
y_preds = np.argmax(test_result.predictions, axis=1)
y_true = test_result.label_ids

print("\n📋 Classification Report:")
print(classification_report(y_true, y_preds, target_names=['negative', 'neutral', 'positive']))

In [ ]:
# 12. Confusion Matrix Visualization
cm = confusion_matrix(y_true, y_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['negative', 'neutral', 'positive'], 
            yticklabels=['negative', 'neutral', 'positive'],
            cbar_kws={'label': 'Count'})
plt.xlabel('Predicted', fontsize=12, fontweight='bold')
plt.ylabel('Actual', fontsize=12, fontweight='bold')
plt.title('Confusion Matrix - IndoBERT 3-Class Sentiment', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Calculate per-class accuracy
print("\n📊 Per-Class Accuracy:")
for i, label in enumerate(['negative', 'neutral', 'positive']):
    class_acc = cm[i, i] / cm[i].sum() * 100
    print(f"   {label:10s}: {class_acc:.2f}%")

In [ ]:
# 13. Save Model
save_path = './saved_model_indobert_3class'

print(f"\n💾 Saving model to {save_path}...")
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("✅ Model saved successfully!")
print("\n📦 Download files:")
print("   - config.json")
print("   - pytorch_model.bin")
print("   - tokenizer files")

In [ ]:
# 14. Test Inference (Quick Test)
print("\n🧪 Testing Inference...\n")

test_texts_sample = [
    "aplikasi bagus banget, driver ramah dan cepat",
    "tidak bisa dipakai, error terus sangat mengecewakan",
    "biasa saja, kadang bagus kadang lambat"
]

# Tokenize
inputs = tokenizer(test_texts_sample, padding=True, truncation=True, max_length=128, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}

# Predict
model.eval()
with torch.no_grad():
    outputs = model(**inputs)
    predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
    predicted_classes = torch.argmax(predictions, dim=-1)

# Show results
for i, text in enumerate(test_texts_sample):
    pred_label = id2label[predicted_classes[i].item()]
    confidence = predictions[i][predicted_classes[i]].item() * 100
    
    print(f"Text: \"{text}\"")
    print(f"Prediction: {pred_label.upper()} (confidence: {confidence:.2f}%)")
    print(f"Probabilities: Neg={predictions[i][0]:.3f}, Neu={predictions[i][1]:.3f}, Pos={predictions[i][2]:.3f}")
    print()

## 🎉 Training Complete!

### Next Steps:
1. Download model dari `saved_model_indobert_3class/`
2. Deploy untuk production
3. Integrate ke aplikasi

### Expected Accuracy:
- **3-Class IndoBERT**: 88-95%
- **F1-Score**: >0.85

### Tips:
- Jika accuracy rendah, coba tambah data training
- Jika overfitting, turunkan learning rate atau tambah regularization
- Monitor confusion matrix untuk identify weak classes